# 🫀 퀘스트 46 · Q7-S — **개인 내 vs 교차환자: 충돌이 아니라 가설이다**

| | **MedKOS / `notebooks/quest46_q7s_personalize.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0067`(Q7-R) · `ailab-2026-0066`(Q7-Q) |
| 규약 | **R22 · R26 ② · R27 ② ③ · R29 ① ② · R30 ① · R31 ④ ⑤ · R32 ② ④ ⑤ · R33 ① ② ⑤ · R34 ① ② ③ ④ ⑤** |
| 학습 | 전역 로지스틱 · GPU 불필요 |

## Q7-R 의 두 결과는 모순이 아니다

| | 결과 |
|---|---|
| **R5** (개체 내 · 층화 · 누출을 키에 넣음) | P 창이 ST 창을 **+0.0974** 로 이김 |
| **R1** (교차환자 LORO · ΔAUPRC) | P 창을 붙이면 **−0.0115** (오히려 손해) |

**추정 대상이 다르다.** R5 는 「**이 사람의** 평소 P vs **이 사람의** 이소성 P」를 가르는
능력이고, R1 은 「**다른 사람에게 옮겨가는** P 형태 특징이 있나」다.

임상적으로도 그럴 법하다. 이소성 초점의 위치는 사람마다 다르고(crista terminalis ·
폐정맥구 · 관상정맥동구 …), 그에 따라 이소성 P 의 축과 모양이 달라진다. 리드 배치·
체형·심장 회전까지 겹친다. **「이소성 P 는 이렇게 생겼다」는 보편 템플릿이 없을 수
있다** — 그래도 각 환자 안에서 「이 사람 평소 P 와 다르다」는 성립한다.

> ### 주 가설 (사전등록)
> **H: 창 형태는 개인 내 판별에 기여하지만 교차환자로 전이되지 않는다.**
>
> 이게 확증되면 Q7 은 **「반년 쓴 부정 결과」에서 「개인화 아키텍처의 근거」**로 성격이 바뀐다.

## 창 설계가 해부학적으로 맞다는 증거 (Q7-R 실측)

```
QRS 개시 중앙   idx 87 = R−36ms
PR 대용치       44샘플 = 122ms (P 피크 → QRS 개시) · IQR [37, 53] = 103~147ms
  → 진짜 PR = +P폭 절반(40~55ms) ≈ **162~177ms**   (교과서 정상 120~200ms 의 정중앙)
P 피크 위치     R−158ms · IQR R−183 ~ −139ms
p_mid_22        R−192 ~ −131ms   ← **P 피크 IQR 이 통째로 안에 들어간다**
```

그리고 실제로 P 창 중 `p_mid_22` 가 가장 강하다. **우연이 아니라 구조적 일관성**이다.

## Q7-R 에서 고칠 것 — 전부 코드로

| | Q7-R 이 틀린 것 | Q7-S 의 처치 |
|---|---|---|
| **검출 규칙** | 출력 어디에도 명시 안 됨 · `raw` 는 회수 +0.0016 인데 ✅ | **규칙을 문자열로 출력** · 검출 = 회수 CI하한 > 0 **AND** 회수 > MDE |
| **추정량 선택** | MDE 단독 → `raw` 선택 | **SNR = 회수/MDE** 를 **실측 효과 근방**(a∈{0.005,0.01,0.02})에서 평균 |
| **주입 격자** | 최소 0.005 인데 raw/lin/rank 는 거기서 이미 ✅ | `a ∈ {0.001 … 0.02}` 로 내린다 |
| **누출 바닥** | 팔 하나 추가에 0.1204→0.1363, 고유항 부호 반전 | **부트스트랩 max 분포** + **팔 개수 민감도**(5·9·13) |
| **`f2_k` 무한회귀** | max-k 를 키에 넣으면 다음 k 가 올라옴 | **`f2_{4..32}` 계열 전체를 한 번에 잔차화** |
| **Δ 의 영점 없음** | −0.0115 가 「정보 없음」인지 「5차원 추가 비용」인지 모름 | **잡음 5차원 · 셔플창 5차원** 대조 팔 |
| **R5 null 미측정** | 3.8배 격차를 인용 못 함 | **셔플 null 20회** 재측정 |

⚠️ **`dr` 은 버그가 아니었다** — Q7-R 에서 정확히 0.5000 이 나온 여섯 팔(`f1`·`f2_16`·
`f2_5`·`f2_10`·`f2_20`·`f4`)은 **전부 확장 기저 안**이라 잔차가 정의상 0 이다(Q7-N 픽스처가
검증한 항등식). 기저 **밖** 프로브에서는 `strat` 대비 누출을 4배 죽였다. 그대로 쓴다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **S1 ★★ (주)** | **Δ(개인화) − Δ(교차환자) > 0** — 같은 창 특징으로 (i) LORO (ii) 환자별 `k` 라벨 미세조정 | 우월성 · Bonf 3 · **레코드 클러스터 부트스트랩** |
| **S2 ★★** | **Δ 의 영점** — `noise5`·`shuf5` 대조 팔의 Δ. P 창 Δ 가 이것들과 **구분되는가** | 짝의 차이 · Bonf 3 |
| **S3 ★★** | **R5 정식화** — `f2` 전 계열 잔차화 + **구성 보장 음성 대조** + **null 재측정** | 우월성 · Bonf 3 |

### 판정표 (R29 ②)

- **S1 ✅ · S2 ✅** → ★★ **H 확증.** 「P 형태는 개인 내에만 있다」 → **개인화 파이프라인의 근거**.
  Q7 을 그 문장으로 종결하고 V 시리즈(소량 라벨 개인화)로 넘긴다
- **S1 ✅ · S2 ❌** → 개인화 이득이 **창 내용이 아니라 차원 추가**에서 온다. 대조 팔이 이겼다는 뜻
- **S1 ❌ · S3 ✅** → 개체 내 신호는 있는데 **소량 라벨로는 못 꺼낸다**. `k` 를 늘리거나 표현을 바꾼다
- **MDE 아래 미결** → ⛔ **측정 한계**(R33 ①). 어떤 분기도 안 탄다

### ★★ 사전등록 종결 조건 (R34 ⑤)

**S1·S3 이 모두 미결이고 S1 의 필요 표본이 300 레코드를 넘으면**,
「**SVDB 규모에서 P 파 고유 신호는 측정 불가능하다**」를 퀘스트 최종 문장으로 쓰고
**형태 갈래를 닫는다. 더 돌지 않는다.**

⚠️ 단 **필요 표본은 이번에 반드시 계산해 출력**한다 — Q7-R 은 R1 MDE 0.0145 로
**124 레코드**면 등가 판정이 되는데 그걸 안 세고 「표본을 늘려야 한다」로만 적었다.
SVDB 78 + MIT-BIH 48 = **126**. **도달 가능한 범위였다.**

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def equiv(lo, hi, margin):
    if lo > -margin and hi < margin: return "✅ 등가"
    if lo > margin or hi < -margin:  return "❌ 차이 있음"
    return "⚠️ 미결"

def mde(lo, hi):
    """★ 최소 검출 효과 = CI 반폭(R33 ①)."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def need_n(n, lo, hi, mean, margin):
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    return None if slack <= 0 else float(n) * (((hi - lo) / 2.0) / slack) ** 2

def judge(mean, lo, hi, n, margin):
    eq, sup = equiv(lo, hi, margin), decide(lo, hi, 0.0, ">")
    nn, m_ = need_n(n, lo, hi, mean, margin), mde(lo, hi)
    if nn is None:
        frame = f"⛔ 등가 불가(점추정 {mean:+.4f} 이 ±{margin} 밖) → 우월성 프레임(R31 ①)"
    elif not np.isfinite(nn):
        frame = "⚠️ 필요 개체 계산 불가"
    else:
        frame = f"등가 필요 **{nn:.0f} 레코드** (현재 {n})"
    limit = "⛔ **MDE 아래 — 측정 한계**" if abs(mean) < m_ else "▸ MDE 위"
    return eq, sup, nn, m_, f"{frame} · MDE **{m_:.4f}** · {limit}"

# ★★ R34 ① — 검출 규칙을 **문자열로 박아 출력**한다. Q7-R 은 규칙이 어디에도 없어서
#    「회수 +0.0016, MDE 0.0300」을 ✅ 로 찍었고 그걸로 추정량을 골랐다.
DETECT_RULE = ("검출 = (회수 CI 하한 > 0) **AND** (회수 점추정 > MDE). "
               "기저의 유의성은 판정에 **들어가지 않는다**.")

def detect(rec_mean, rec_lo, m_):
    """회수(주입 − 기저)로만 판정한다. 기저가 이미 유의해도 그건 검출이 아니다."""
    if not (np.isfinite(rec_mean) and np.isfinite(rec_lo) and np.isfinite(m_)):
        return False
    return bool(rec_lo > 0 and rec_mean > m_)

def snr_select(rows, target_a):
    """★★ R34 ② — 추정량 선택은 **SNR = 회수/MDE** 를 **실측 효과 근방**에서 평균한다.

    MDE 단독은 스케일 의존이라 못 쓴다 — `strat` 은 칸 안에서 리듬 분산을 지우므로 같은
    주입이 더 크게 보인다. Q7-R 실측에서 순위가 진폭에 따라 뒤집혔다
    (a=0.02 는 strat 0.648 > raw 0.467, a=0.05 는 raw 1.780 > strat 1.546)."""
    out = {}
    for m, d in rows.items():
        v = [d["rec"][a] / d["mde"] for a in target_a
             if a in d["rec"] and np.isfinite(d["rec"][a]) and d["mde"] > 0]
        out[m] = float(np.mean(v)) if v else float("nan")
    ok = {m: v for m, v in out.items() if np.isfinite(v)}
    return out, (max(ok, key=ok.get) if ok else None)

class AssetError(RuntimeError): pass
print("CELL 0 ✅ ·", DETECT_RULE)

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S, RPRE, NB_BOOT = 20260804, 1, 100, 4000

# ── 창 (Q7-Q/R 승계 · 폭 22 · 겹침 없음)
W22 = 22
SEGS22 = {"p_early_22": (0, 22), "p_mid_22": (31, 53),
          "p_late_strict": (53, 75), "stt_22": (130, 152)}
GATE_WIN, NEG_WIN = "p_mid_22", "stt_22"

# ── ★★ S1 개인화 대조
K_PERS = (5, 10, 20)             # 환자별 라벨 비트 수(**클래스당**)
N_PCA, SPEC_LO = 5, 0.95
EQ_DELTA = 0.01                  # 사전등록 등가 임계(ΔAUPRC)

# ── ★★ S2 Δ 의 영점 — 대조 팔 둘
#    `noise5` 순수 가우시안 5차원 · `shuf5` 창 PCA 5차원을 **레코드 안에서 치환**
#    (주변분포 동일 · 라벨 관계만 끊음). Q7-R 은 이게 없어 −0.0115 를 해석 못 했다.
CTRL_ARMS = ("noise5", "shuf5")

# ── ★★ S3 `f2_k` 무한회귀 종결 — 계열 **전체**를 한 번에 잔차화한다
FULL_K   = tuple(range(4, 33))   # f2_4 … f2_32 전부 기저에
PROBE_K  = (40, 48, 56)          # ★ 계열 **밖** 프로브 — 안 그러면 검증이 안 된다
LB_K, F2_BIN, TREND_W = 16, 0.02, 8
MIN_S_TPL, K_FOLD, N_REPEAT, N_SHUF = 20, 5, 3, 20

# ── ★★ R34 ① ② 교정
AMPS      = (0.001, 0.002, 0.003, 0.005, 0.010, 0.020)   # ★ 아래로 확장(R33 ②)
TARGET_A  = (0.005, 0.010, 0.020)                        # ★ 실측 효과 a≈0.008 근방
HETERO    = 0.35
LEAK_SUBSET_SIZES = (5, 9, 13)   # ★ 누출 바닥의 **팔 개수 민감도**(선택 편의 진단)

LADDER  = ("raw", "lin", "rank", "strat", "dr")
PRIMARY = "strat"                # 잠정 — S 교정 블록이 다음 실행의 주 추정량을 정한다
BONF3   = 0.05 / 3 / 2           # 1차 가족 {S1 · S2 · S3}

RULE_CHECK = {
    "R16 fallback 없음":        "자산 셀에서 예외 삼킴 없음",
    "R22 교차적합":             "개체 내 겹 밖 · 전역은 LORO · **개인화는 쓴 비트를 평가에서 뺀다**",
    "R26 ② 자기 null":          "라벨셔플 null 위 초과분 · SE 전파",
    "R27 ③ 폭 정합":            "폭 22 계열 동일 · 겹침 없음(코드 강제)",
    "R29 ② 측정 불가 분기 금지": "⛔ 판정은 어떤 결론 분기도 타지 않는다",
    "R31 ① 등가/우월 둘 다":    "judge() 가 둘 다 + 필요표본 + MDE",
    "R32 ④ 누출 바닥 양수":     "양의 초과 max · 음의 초과는 분리",
    "R32 ⑤ 바닥은 수준에만":    "짝의 차이에서 안 뺀다",
    "R33 ① 관문 통과형 대조":   "주입을 관문에 그대로 통과",
    "R33 ② 격자가 바닥을 감싼다": "★ a=0.001 까지 내린다",
    "R33 ⑤ 리듬 대비 임계":     "ΔAUPRC ±0.01 · 천장효과 제거",
    "R34 ① 검출 규칙 명시":     "★★ DETECT_RULE 을 **문자열로 출력** · 회수로만 판정",
    "R34 ② 선택은 SNR":         "★★ 회수/MDE 를 **실측 효과 근방**에서 평균",
    "R34 ③ 구성 보장 음성대조": "★★ 같은 레코드의 **무작위 다른 비트** 창",
    "R34 ④ 문턱 근거":          "★ 하드코딩 분기 없음 — 전부 CI/MDE 에서 유도",
    "R34 ⑤ 종결 조건":          "★★ S1·S3 미결 + 필요표본 >300 → 형태 갈래를 닫는다",
}

CONFIG = dict(
    exp="quest46_q7s_personalize", quest="ailab-2026-0046", step="svdb-personalize",
    parent_exp=["quest46_q7r_calibrate", "ailab-2026-0067"],
    purpose=("Q7-R 의 R1(교차환자 ΔAUPRC −0.0115)과 R5(개체 내 P−ST +0.0974)는 모순이 "
             "아니라 **추정 대상이 다르다**. 이소성 초점 위치가 사람마다 달라 「이소성 P 는 "
             "이렇게 생겼다」는 보편 템플릿이 없을 수 있고, 그래도 각 환자 안에서 「이 사람 "
             "평소 P 와 다르다」는 성립한다. 그걸 **주 가설로 승격**한다 — H: 창 형태는 "
             "개인 내 판별에 기여하지만 교차환자로 전이되지 않는다. 확증되면 Q7 은 부정 "
             "결과가 아니라 **개인화 아키텍처의 근거**가 된다. 동시에 Q7-R 의 교정 결함 "
             "넷(검출 규칙 · 선택 기준 · 격자 · 누출 바닥 선택편의)을 코드로 고친다"),
    dataset="SVDB 전수 · svdb_data5.npz + Q7-B 예측 캐시(라벨·매핑용)",
    windows={k: list(v) for k, v in SEGS22.items()}, gate_win=GATE_WIN, neg_win=NEG_WIN,
    k_pers=list(K_PERS), ctrl_arms=list(CTRL_ARMS), full_k=list(FULL_K),
    probe_k=list(PROBE_K), amps=list(AMPS), target_a=list(TARGET_A),
    leak_subset_sizes=list(LEAK_SUBSET_SIZES), detect_rule=DETECT_RULE,
    eq_delta=EQ_DELTA, ladder=list(LADDER), rule_check=RULE_CHECK,
    predictions={
        "S1": "★★ **주 관문** — Δ(개인화 `k` 라벨) − Δ(교차환자 LORO) > 0. 같은 창 특징 · "
              "같은 평가 비트(개인화에 쓴 비트는 **양쪽 다** 평가에서 뺀다). Bonferroni 3",
        "S2": "★★ **Δ 의 영점** — `noise5`(순수 잡음 5차원) · `shuf5`(창 PCA 를 레코드 안에서 "
              "치환) 대조 팔. P 창 Δ 가 이것들과 **구분되는가**. 구분 안 되면 −0.0115 는 "
              "「정보 없음」이 아니라 **차원 추가의 0차 비용**이다",
        "S3": "★★ **R5 정식화** — `f2_{4..32}` 계열 **전체** 잔차화 + **구성 보장 음성 대조** "
              "+ 셔플 null **재측정**. Q7-R 은 null 을 안 재서 3.8배 격차를 인용 못 했다",
        "S4": "(교정) 검출 규칙 **출력** · 격자 a=0.001 까지 · **SNR 선택** · 누출 바닥 "
              "**부트스트랩 max + 팔 개수 민감도**",
        "S5": "(계산) **필요 표본을 반드시 출력**. Q7-R 은 R1 MDE 0.0145 로 124 레코드면 "
              "되는데 안 세고 「표본을 늘려야」로만 적었다"},
    caveat=("★ **`dr` 은 버그가 아니었다** — Q7-R 의 0.5000 여섯 팔은 전부 확장 기저 안이라 "
            "잔차가 정의상 0 이다. 기저 밖 프로브에서 누출을 4배 죽였다. 그대로 쓴다. "
            "★ **개인화는 상한이다** — 같은 레코드에서 라벨을 가져오므로 배포 시나리오의 "
            "최선이다. 그래도 (i)/(ii) 비교는 **같은 평가 비트**에서 하므로 대조는 공정하다. "
            "★ **`shuf5` 는 레코드 안 치환**이라 주변분포는 같고 라벨 관계만 끊는다. "
            "★ 창 특징은 PCA 라 「무엇이 기여했나」를 말하지 않는다 — 「기여가 있나」만 묻는다. "
            "학습은 전역 로지스틱뿐 · GPU 불필요 · 예상 40~55분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7s_personalize", CONFIG, project=PROJECT)
run.log("설정 ✅ 주 가설 — **창 형태는 개인 내에만 있고 교차환자로 전이되지 않는다**")
run.log(f"  ★ 검출 규칙: {DETECT_RULE}")
run.log(f"  ★ 주입 격자 {AMPS} · SNR 평가 구간 {TARGET_A}")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<24} {v_}")

In [ ]:
# CELL 2 — 【S-0a】 자산 · 매핑 (fallback 없음 R16)
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
SV5  = os.path.join(MITBIH, "svdb_data5.npz")
for p_, why in ((PROB, "Q7-B 예측 캐시"), (SV5, "SVDB 빌드")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}")
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb

P = dict(np.load(PROB))
Y = np.asarray(P["y_cross"]); REC = np.asarray(P["rec_cross"]).astype(int)
recs = [int(r) for r in wfdb.get_record_list("svdb")]
if len(recs) < 2:
    raise AssetError("SVDB 목록을 못 받았다")
labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError("라벨이 목록 밖인데 연속 가정과도 안 맞는다")
    MAP = {l: recs[l - CONT[0]] for l in labels}
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)

d5 = np.load(SV5, allow_pickle=True)
keep = d5["y3"] >= 0
assert int(keep.sum()) == len(Y), f"길이 불일치 {int(keep.sum())} vs {len(Y)}"
PRE  = np.asarray(d5["pre_rr"])[keep].astype(float)
POST = np.asarray(d5["post_rr"])[keep].astype(float)
BEAT = np.ascontiguousarray(np.asarray(d5["beat"])[keep]).astype("float32")
ALLR = [int(r) for r in np.unique(REC)]
run.log("\n" + "=" * 100)
run.log("【S-0a】 자산 · 창")
run.log("=" * 100)
run.log(f"  예측 {len(Y):,}비트 · 레코드 {len(ALLR)}개 · 유병률 {float((Y==IDX_S).mean()):.4f}")
_w = {k: v[1] - v[0] for k, v in SEGS22.items()}
if len(set(_w.values())) != 1:
    raise AssetError(f"폭이 다르다 {_w} — R27 ③ 위반")
_o = sorted(SEGS22.values())
for (a1, b1), (a2, b2) in zip(_o, _o[1:]):
    if b1 > a2:
        raise AssetError(f"창이 겹친다 {(a1,b1)} vs {(a2,b2)}")
for k_, (a_, b_) in SEGS22.items():
    run.log(f"    {k_:<14} idx {a_:>3}–{b_:<3} = R{(a_-RPRE)/360*1000:+.0f}~"
            f"{(b_-RPRE)/360*1000:+.0f}ms")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【S-A】 특징(f2 전 계열) · 층 키 · 구성 보장 음성 대조 · 점수
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

def local_base(pre_v, k):
    n = len(pre_v); out = np.empty(n); med = float(np.median(pre_v))
    for i in range(n):
        a = max(0, i - k)
        out[i] = med if i - a < 3 else float(np.median(pre_v[a:i]))
    return out

def trend(pre_v, w):
    n = len(pre_v); out = np.zeros(n); x = np.arange(w, dtype=float)
    xc = x - x.mean(); den = float((xc * xc).sum())
    for i in range(n):
        a = i - w
        if a < 0: continue
        y = pre_v[a:i]
        out[i] = float(((y - y.mean()) * xc).sum() / den)
    return out

ALL_K = sorted(set(FULL_K) | set(PROBE_K) | {LB_K})

def all_feats(pre_v, post_v):
    med = float(np.median(pre_v)); n = len(pre_v)
    F = {"f1": med - pre_v}
    for k in ALL_K:
        F[f"f2_{k}"] = 1.0 - pre_v / np.maximum(local_base(pre_v, k), 1e-9)
    F["f6"] = 1.0 - local_base(pre_v, LB_K) / max(med, 1e-9)
    cv = np.empty(n)
    for i in range(n):
        a = max(0, i - TREND_W); w_ = pre_v[a:i] if i - a >= 3 else pre_v[:3]
        cv[i] = float(np.std(w_) / max(np.mean(w_), 1e-9))
    F["f4"] = cv
    F["trend"] = trend(pre_v, TREND_W)
    F["f5"] = np.r_[0.0, F[f"f2_{FULL_K[0]}"][:-1]]
    F["f3"] = 1.0 - (pre_v + post_v) / np.maximum(2.0 * local_base(pre_v, FULL_K[0]), 1e-9)
    F["f1_rank"] = stats.rankdata(F["f1"]) / n
    return F

def basis_mat(F, kind):
    """★★ `ext` 가 **`f2_{4..32}` 계열 전체**를 담는다 — max-k 를 쫓는 무한회귀를 끝낸다
    (Q7-R 은 f2_5 를 키에 넣었더니 f2_7 이 올라왔다). 프로브는 계열 **밖**(k=40·48·56)."""
    cols = ["f1", "f6"]
    if kind == "ext":
        cols = cols + [f"f2_{k}" for k in FULL_K] + ["f4", "trend"]
    else:
        cols = cols + [f"f2_{k}" for k in (8, 16, 32)]
    Z = np.stack([F[c] for c in cols], axis=1)
    if kind == "rank":
        Z = np.stack([stats.rankdata(F[c]) / len(F[c]) for c in cols], axis=1)
    elif kind not in ("lin", "ext"):
        raise AssetError(f"기저 종류 {kind} 를 모른다")
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-12)
    return np.c_[np.ones(len(Z)), Z]

def prep(s, kind):
    s = np.asarray(s, float)
    return stats.rankdata(s) / len(s) if kind == "rank" else s

def residualize(s, Z):
    s = np.asarray(s, float)
    if not np.isfinite(s).all(): return None
    beta, *_ = np.linalg.lstsq(Z, s, rcond=None)
    e = s - Z @ beta
    return np.zeros_like(e) if float(np.std(e)) <= 1e-9 * (float(np.std(s)) + 1e-12) else e

def strat_key(F):
    pk = np.unique(np.round(F["f1"], 6), return_inverse=True)[1].astype(np.int64)
    f2b = np.floor(F[f"f2_{LB_K}"] / F2_BIN).astype(np.int64)
    return pk * (int(f2b.max() - f2b.min()) + 1) + (f2b - f2b.min())

def strat_auc(sc, tt, key):
    sc = np.asarray(sc, float)
    if not np.isfinite(sc).all(): return float("nan")
    uq, inv = np.unique(key, return_inverse=True)
    num = den = 0.0
    for j in range(len(uq)):
        m = inv == j
        s_, n_ = sc[m & tt], sc[m & ~tt]
        if not len(s_) or not len(n_): continue
        num += float((s_[:, None] > n_[None, :]).sum()) + 0.5 * float((s_[:, None] == n_[None, :]).sum())
        den += float(len(s_) * len(n_))
    return num / den if den >= 1 else float("nan")

def dist(Bw, ref):
    dd = Bw - ref[None]
    return np.sqrt((dd * dd).sum(axis=(1, 2)))

def two_template_cv(Bw, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            sc[te] = dist(Bw[te], np.median(Bw[tr & ~tt], axis=0)) \
                   - dist(Bw[te], np.median(Bw[tr & tt], axis=0))
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

def cv_logit(X, tt, K, seed, n_rep):
    accs = []
    for rep in range(max(n_rep, 1)):
        rng = np.random.RandomState(seed + 1000 * rep)
        fold = rng.permutation(len(tt)) % K
        sc = np.full(len(tt), np.nan)
        for f in range(K):
            te = fold == f; tr = ~te
            if not te.any() or int(tt[tr].sum()) < 2 or int((~tt[tr]).sum()) < 2:
                return None
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=2000, C=1.0)
            lr.fit((X[tr] - mu) / sd, tt[tr].astype(int))
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        accs.append(sc)
    return np.mean(np.stack(accs), axis=0)

RHY_COLS = ["f1", "f2_8", "f2_16", "f3", "f4", "f5", "f6"]

run.log("\n" + "=" * 100)
run.log("【S-A】 특징 · 층 키 · 구성 보장 음성 대조")
run.log("=" * 100)
T0 = time.time()
FEAT, META, SKIP = {}, {}, []
for r in ALLR:
    mm = np.where(REC == r)[0]
    tt = (Y[mm] == IDX_S)
    if int(tt.sum()) < MIN_S_TPL or int((~tt).sum()) < MIN_S_TPL:
        SKIP.append(int(r)); continue
    F = all_feats(PRE[mm], POST[mm])
    prv = np.r_[False, tt[:-1]]; nxt = np.r_[tt[1:], False]
    FEAT[int(r)] = F
    META[int(r)] = dict(tt=tt, idx=mm, key=strat_key(F), n=len(mm), pos=int(tt.sum()),
                        amp=float(np.median(BEAT[mm][:, :, 85:115].max(axis=2)
                                            - BEAT[mm][:, :, 85:115].min(axis=2))),
                        iso=float((tt & ~prv & ~nxt).sum() / max(tt.sum(), 1)))
RS = sorted(META)
run.log(f"  채점 **{len(RS)}개체** · 제외 {len(SKIP)} · {time.time()-T0:.0f}초")
run.log(f"  기저에 `f2_k` **전 계열** k={FULL_K[0]}..{FULL_K[-1]} ({len(FULL_K)}개) — "
        f"무한회귀 종결(R34 의 f2_k 문제)")
run.log(f"  프로브는 계열 **밖** k={PROBE_K}")
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【S-B】 창 · 구성 보장 음성 대조 · 점수 · 추정량 채점
def win_arr(r, name, inject=None, shuffle_neg=False):
    """창 파형. `shuffle_neg=True` 면 **같은 레코드의 무작위 다른 비트**에서 가져온다.

    ★★ R34 ③ — 「해부학적으로 무관해 보이는 창」은 음성 대조가 아니었다(Q7-R 에서
    `stt_22` 가 교차환자 1등). **구성으로 보장**한다: 형태 통계는 그대로이고 라벨 관계만
    끊긴다."""
    a, b = SEGS22[name]
    Bm = BEAT[META[r]["idx"]]
    Bw = Bm[:, :, a:b].astype("float64")
    if shuffle_neg:
        rng = np.random.RandomState(SEED0 + 613 + int(r))
        Bw = Bw[rng.permutation(len(Bw))].copy()
    else:
        Bw = Bw.copy()
    if inject is not None:
        amp, rng = inject
        tt = META[r]["tt"]; L = b - a; x = np.arange(L, dtype=float)
        c = (L - 1) / 2.0 * (1.0 + HETERO * rng.uniform(-1, 1))
        w = max(L / 6.0 * (1.0 + HETERO * rng.uniform(-1, 1)), 1.0)
        sgn = 1.0 if rng.uniform() > 0.25 else -1.0
        Bw[tt] += sgn * amp * META[r]["amp"] * np.exp(-((x - c) ** 2) / (2 * w ** 2))[None, None, :]
    return Bw

NEG_SHUF = "neg_shuf"            # ★ 구성 보장 음성 대조
ARMS_WIN = list(SEGS22) + [NEG_SHUF]
PROBES = [f"f2_{k}" for k in PROBE_K] + ["f1_rank"]
ARMS = ["f1", "f2_16", "f3", "f4", "f5"] + PROBES + ARMS_WIN + ["lr_rhy"]

def build_scores(r, tt, K, seed, inject=None):
    S = {}
    F = FEAT[r]
    X = np.stack([F[c] for c in RHY_COLS], axis=1)
    sc = cv_logit(X, tt, K, seed, N_REPEAT)
    if sc is None: return None
    S["lr_rhy"] = sc
    for nm_ in ARMS_WIN:
        src = GATE_WIN if nm_ == NEG_SHUF else nm_
        inj = inject if (inject is not None and nm_ == GATE_WIN) else None
        st = two_template_cv(win_arr(r, src, inject=inj, shuffle_neg=(nm_ == NEG_SHUF)),
                             tt, K, seed, N_REPEAT)
        S[nm_] = st if st is not None else np.full(len(tt), np.nan)
    return S

def score_of(r, a, S):
    return S[a] if a in S else FEAT[r][a]

run.log("\n" + "=" * 100)
run.log("【S-B】 추정량 5종 — " + " · ".join(LADDER) +
        f"  (★ 구성 보장 음성 대조 `{NEG_SHUF}` 포함)")
run.log("=" * 100)
MET = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
SCORES = {}
T0 = time.time()
for i, r in enumerate(RS):
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    S = build_scores(r, tt, K_FOLD, SEED0)
    if S is None: continue
    SCORES[r] = S
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    for a in ARMS:
        s = score_of(r, a, S)
        if not np.isfinite(s).all(): continue
        MET["raw"][a][i] = roc_auc_score(tt.astype(int), s)
        for b_ in ("lin", "rank"):
            e = residualize(prep(s, b_), Z[b_])
            if e is not None: MET[b_][a][i] = roc_auc_score(tt.astype(int), e)
        MET["strat"][a][i] = strat_auc(s, tt, key)
        e = residualize(s, Z["ext"])
        if e is not None: MET["dr"][a][i] = strat_auc(e, tt, key)
run.log(f"  ({time.time()-T0:.0f}초) 팔별 — " + " · ".join(LADDER))
for a in ARMS:
    star = "  ★" if a in (GATE_WIN, NEG_WIN, NEG_SHUF) else ("  ▸" if a in PROBES else "")
    run.log(f"    {a:<14} " + " · ".join(f"{np.nanmean(MET[m][a]):.4f}" for m in LADDER) + star)
CONFIG["metrics"] = {m: {a: float(np.nanmean(MET[m][a])) for a in ARMS} for m in LADDER}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【S-C】 라벨셔플 null (전 팔 · 5종 · R26 ②) — ★ R5 가 못 쟀던 그 null
run.log("\n" + "=" * 100)
run.log(f"【S-C】 라벨셔플 null (셔플 {N_SHUF}회 · {len(RS)}개체)")
run.log("=" * 100)
T1 = time.time()
NULL = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
NSE  = {m: {a: np.full(len(RS), np.nan) for a in ARMS} for m in LADDER}
for i, r in enumerate(RS):
    if r not in SCORES: continue
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    acc = {m: {a: [] for a in ARMS} for m in LADDER}
    for s_ in range(N_SHUF):
        rng = np.random.RandomState(SEED0 + 7919 * (s_ + 1) + int(r))
        ts = rng.permutation(tt)
        Ss = build_scores(r, ts, K_FOLD, SEED0 + 31 * (s_ + 1))
        if Ss is None: continue
        for a in ARMS:
            s = score_of(r, a, Ss)
            if not np.isfinite(s).all(): continue
            acc["raw"][a].append(roc_auc_score(ts.astype(int), s))
            for b_ in ("lin", "rank"):
                e = residualize(prep(s, b_), Z[b_])
                if e is not None: acc[b_][a].append(roc_auc_score(ts.astype(int), e))
            acc["strat"][a].append(strat_auc(s, ts, key))
            e = residualize(s, Z["ext"])
            if e is not None: acc["dr"][a].append(strat_auc(e, ts, key))
    for m in LADDER:
        for a in ARMS:
            v_ = np.asarray([x for x in acc[m][a] if np.isfinite(x)], float)
            if len(v_) >= 2:
                NULL[m][a][i] = float(v_.mean())
                NSE[m][a][i] = float(v_.std(ddof=1) / np.sqrt(len(v_)))
run.log(f"  ({time.time()-T1:.0f}초)  초과분 (실측 − null)")
for a in ARMS:
    star = "  ★" if a in (GATE_WIN, NEG_WIN, NEG_SHUF) else ("  ▸" if a in PROBES else "")
    run.log(f"    {a:<14} " + " · ".join(
        f"{m} {np.nanmean(MET[m][a])-np.nanmean(NULL[m][a]):+.4f}" for m in LADDER) + star)
CONFIG["excess"] = {m: {a: float(np.nanmean(MET[m][a]) - np.nanmean(NULL[m][a]))
                        for a in ARMS} for m in LADDER}
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【S-D】 S4 교정 — 검출 규칙 · SNR 선택 · 누출 바닥(부트스트랩+팔개수)
def boot_pair(a1, a2, meth, seed, nb=NB_BOOT, q=2.5, arr=None):
    m1 = MET[meth][a1] if arr is None else arr
    d = (m1 - NULL[meth][a1]) - (MET[meth][a2] - NULL[meth][a2])
    se = np.sqrt(np.nan_to_num(NSE[meth][a1]) ** 2 + np.nan_to_num(NSE[meth][a2]) ** 2)
    ok = np.isfinite(d); d, se = d[ok], se[ok]
    if len(d) < 3: return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed); v = np.empty(nb)
    for b in range(nb):
        ix = rng.randint(0, len(d), len(d))
        v[b] = (d[ix] - rng.normal(0.0, 1.0, len(ix)) * se[ix]).mean()
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

def boot_rec_diff(vec, seed, nb=2000, q=2.5):
    """개체별 값 벡터의 평균과 CI(레코드 부트스트랩)."""
    d = vec[np.isfinite(vec)]
    if len(d) < 3: return float("nan"), float("nan"), float("nan"), 0
    rng = np.random.RandomState(seed)
    v = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return float(d.mean()), float(np.percentile(v, q)), float(np.percentile(v, 100 - q)), len(d)

run.log("\n" + "=" * 100)
run.log("【S-D】 S4 교정")
run.log("=" * 100)
run.log(f"  ★★ **검출 규칙**: {DETECT_RULE}")
T2 = time.time()
INJ = {m: {a_: np.full(len(RS), np.nan) for a_ in AMPS} for m in LADDER}
for i, r in enumerate(RS):
    if r not in SCORES: continue
    tt = META[r]["tt"]; F = FEAT[r]; key = META[r]["key"]
    Z = {b_: basis_mat(F, b_) for b_ in ("lin", "rank", "ext")}
    for a_ in AMPS:
        rng = np.random.RandomState(SEED0 + 977 * int(r) + int(a_ * 100000))
        st = two_template_cv(win_arr(r, GATE_WIN, inject=(a_, rng)), tt, K_FOLD,
                             SEED0, N_REPEAT)
        if st is None: continue
        INJ["raw"][a_][i] = roc_auc_score(tt.astype(int), st)
        for b_ in ("lin", "rank"):
            e = residualize(prep(st, b_), Z[b_])
            if e is not None: INJ[b_][a_][i] = roc_auc_score(tt.astype(int), e)
        INJ["strat"][a_][i] = strat_auc(st, tt, key)
        e = residualize(st, Z["ext"])
        if e is not None: INJ["dr"][a_][i] = strat_auc(e, tt, key)
run.log(f"  ({time.time()-T2:.0f}초)")

ROWS = {}
run.log(f"\n  추정량별 — 관문 `{GATE_WIN} − {NEG_SHUF}`(구성 보장 음성 대조)")
for m in LADDER:
    bm, blo, bhi, bn = boot_pair(GATE_WIN, NEG_SHUF, m, SEED0 + 11, q=BONF3 * 100)
    m_ = mde(blo, bhi)
    rec, hit, row = {}, None, []
    for a_ in AMPS:
        rm, rlo, rhi, _ = boot_pair(GATE_WIN, NEG_SHUF, m, SEED0 + 21, q=BONF3 * 100,
                                    arr=INJ[m][a_])
        # ★★ 회수(주입 − 기저)로만 판정한다 — 기저의 유의성은 안 본다(R34 ①)
        rc = rm - bm; rc_lo = rlo - bm
        rec[a_] = rc
        det = detect(rc, rc_lo, m_)
        row.append(f"a={a_:.3f} 회수 {rc:+.4f}{'✅' if det else '❌'}")
        if hit is None and det: hit = a_
    ROWS[m] = dict(base=bm, lo=blo, hi=bhi, n=bn, mde=m_, rec=rec, floor=hit)
    run.log(f"    {m:<6} 기저 {bm:+.4f} [{blo:+.4f}, {bhi:+.4f}] · MDE **{m_:.4f}** · "
            f"바닥 {('a≥%.3f' % hit) if hit else '⛔ 격자 안 미검출'}")
    run.log(f"           " + " | ".join(row))
SNR, BEST = snr_select(ROWS, TARGET_A)
run.log(f"\n  ★★ **SNR = 회수/MDE** (실측 효과 근방 a∈{TARGET_A} 평균 · R34 ②)")
for m in LADDER:
    run.log(f"    {m:<6} SNR {SNR[m]:.4f}" + ("   ← **최대**" if m == BEST else ""))
run.log(f"    ★ **다음 실행 주 추정량 = `{BEST}`** — MDE 단독이 아니라 SNR 로 골랐다")
BRACKET = any(ROWS[m]["floor"] is None or ROWS[m]["floor"] > AMPS[0] for m in LADDER)
run.log(f"    ★ 격자 감쌈 — {'✅' if BRACKET else '⛔ **미확정**(최소점에서도 전부 검출 · R33 ②)'}")
CONFIG["estimator"] = {m: dict(mde=float(ROWS[m]["mde"]), snr=float(SNR[m]),
                               floor=(None if ROWS[m]["floor"] is None else float(ROWS[m]["floor"])))
                       for m in LADDER}
CONFIG["best_estimator"] = BEST; CONFIG["bracketed"] = bool(BRACKET)

# ── ★ 누출 바닥 — 부트스트랩 max + **팔 개수 민감도**(선택 편의 진단)
run.log(f"\n  누출 바닥 [{PRIMARY}] — ★ 부트스트랩 max + **팔 개수 민감도**")
LEAK_POOL = PROBES + ["f3", "f4", "f5", "f1", "f2_16"]
EXC = {a: MET[PRIMARY][a] - NULL[PRIMARY][a] for a in LEAK_POOL}
def leak_boot(arms, seed, nb=2000):
    """개체 부트스트랩 안에서 **팔별 평균의 max** 분포. 점추정 max 는 선택 편의가 있다."""
    idx = np.arange(len(RS)); rng = np.random.RandomState(seed); v = []
    for _ in range(nb):
        ix = rng.randint(0, len(idx), len(idx))
        mx = max(float(np.nanmean(EXC[a][ix])) for a in arms)
        if np.isfinite(mx): v.append(mx)
    pt = max(float(np.nanmean(EXC[a])) for a in arms)
    return pt, float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
for sz in LEAK_SUBSET_SIZES:
    sub = LEAK_POOL[:min(sz, len(LEAK_POOL))]
    pt, lo_, hi_ = leak_boot(sub, SEED0 + 31 + sz)
    run.log(f"    팔 {len(sub):>2}개 → 바닥 점추정 **{pt:+.4f}** [{lo_:+.4f}, {hi_:+.4f}]")
run.log("    ▸ 팔을 늘릴수록 점추정이 **단조 증가**하면 그건 신호가 아니라 **선택 편의**다")
LEAK_PT, LEAK_LO, LEAK_HI = leak_boot(LEAK_POOL, SEED0 + 41)
CONFIG["leak"] = dict(pt=LEAK_PT, lo=LEAK_LO, hi=LEAK_HI)
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【S-E】 ★★ S1 개인화 대조 · S2 Δ 의 영점 (주 관문)
def partial_auc(y, s, spec_lo):
    fpr, tpr, _ = roc_curve(y, s)
    hi = 1.0 - spec_lo
    m = fpr <= hi
    if m.sum() < 2: return float("nan")
    x, yv = fpr[m], tpr[m]
    if x[-1] < hi:
        x = np.r_[x, hi]; yv = np.r_[yv, np.interp(hi, fpr, tpr)]
    return float((np.diff(x) * (yv[:-1] + yv[1:]) / 2.0).sum()) / hi   # np.trapz 는 numpy2 에서 제거

run.log("\n" + "=" * 100)
run.log("【S-E】 ★★ S1 개인화 대조 · S2 Δ 의 영점")
run.log("=" * 100)
T3 = time.time()
RHY_ALL = np.concatenate([np.stack([FEAT[r][c] for c in RHY_COLS], 1) for r in RS])
TT_ALL  = np.concatenate([META[r]["tt"] for r in RS])
RID     = np.concatenate([np.full(META[r]["n"], r) for r in RS])
WIN_RAW = {w: np.concatenate([BEAT[META[r]["idx"]][:, :, SEGS22[w][0]:SEGS22[w][1]]
                              .reshape(META[r]["n"], -1) for r in RS]) for w in SEGS22}
_rng = np.random.RandomState(SEED0 + 99)
NOISE5 = _rng.normal(size=(len(TT_ALL), N_PCA))      # ★ S2 — 순수 잡음 5차원

TEST_ARMS = [GATE_WIN, NEG_WIN] + list(CTRL_ARMS)

def feats_for(arm, tr, te):
    """학습 레코드에서만 적합한 창 특징. `shuf5` 는 **레코드 안에서 치환**해 라벨 관계만 끊는다."""
    if arm == "noise5":
        return NOISE5[tr], NOISE5[te]
    src = GATE_WIN if arm == "shuf5" else arm
    pca = PCA(n_components=N_PCA, random_state=SEED0).fit(WIN_RAW[src][tr])
    Ztr, Zte = pca.transform(WIN_RAW[src][tr]), pca.transform(WIN_RAW[src][te])
    if arm == "shuf5":
        rr = np.random.RandomState(SEED0 + 7)
        for Z_, m_ in ((Ztr, tr), (Zte, te)):
            for u in np.unique(RID[m_]):
                sel = np.where(RID[m_] == u)[0]
                Z_[sel] = Z_[sel][rr.permutation(len(sel))]
    return Ztr, Zte

def fit_eval(Xtr, ytr, Xte, yte):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((Xtr - mu) / sd, ytr)
    s = lr.decision_function((Xte - mu) / sd)
    return average_precision_score(yte, s), partial_auc(yte, s, SPEC_LO)

# 개체별 Δ = AUPRC(리듬+창) − AUPRC(리듬).  (i) LORO  (ii) 개인화 k
D_LORO = {a: np.full(len(RS), np.nan) for a in TEST_ARMS}
D_PERS = {k: {a: np.full(len(RS), np.nan) for a in TEST_ARMS} for k in K_PERS}
DP_LORO = {a: np.full(len(RS), np.nan) for a in TEST_ARMS}
for i, r in enumerate(RS):
    te0 = np.where(RID == r)[0]
    s_idx = te0[TT_ALL[te0]]; n_idx = te0[~TT_ALL[te0]]
    if len(s_idx) < max(K_PERS) + 5 or len(n_idx) < max(K_PERS) + 5:
        continue
    rng = np.random.RandomState(SEED0 + 313 + int(r))
    pick = {k: (rng.choice(s_idx, k, replace=False), rng.choice(n_idx, k, replace=False))
            for k in K_PERS}
    used = np.unique(np.concatenate([np.r_[a, b] for a, b in pick.values()]))
    # ★ (i)/(ii) 를 **같은 평가 비트**에서 비교한다 — 개인화에 쓴 비트는 양쪽 다 뺀다
    ev = np.setdiff1d(te0, used)
    tr0 = np.where(RID != r)[0]
    if len(ev) < 30 or TT_ALL[ev].sum() < 3:
        continue
    base_i = fit_eval(RHY_ALL[tr0], TT_ALL[tr0].astype(int), RHY_ALL[ev],
                      TT_ALL[ev].astype(int))
    base_p = {}
    for k in K_PERS:
        tr_k = np.r_[tr0, pick[k][0], pick[k][1]]
        base_p[k] = fit_eval(RHY_ALL[tr_k], TT_ALL[tr_k].astype(int), RHY_ALL[ev],
                             TT_ALL[ev].astype(int))
    for a in TEST_ARMS:
        Ztr, Zev = feats_for(a, tr0, ev)
        f_i = fit_eval(np.c_[RHY_ALL[tr0], Ztr], TT_ALL[tr0].astype(int),
                       np.c_[RHY_ALL[ev], Zev], TT_ALL[ev].astype(int))
        D_LORO[a][i] = f_i[0] - base_i[0]; DP_LORO[a][i] = f_i[1] - base_i[1]
        for k in K_PERS:
            tr_k = np.r_[tr0, pick[k][0], pick[k][1]]
            Ztr2, Zev2 = feats_for(a, tr_k, ev)
            f_p = fit_eval(np.c_[RHY_ALL[tr_k], Ztr2], TT_ALL[tr_k].astype(int),
                           np.c_[RHY_ALL[ev], Zev2], TT_ALL[ev].astype(int))
            D_PERS[k][a][i] = f_p[0] - base_p[k][0]
run.log(f"  ({time.time()-T3:.0f}초)")

VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

run.log(f"\n  ΔAUPRC (리듬 전용 대비 · 개체별 평균)")
run.log(f"    {'팔':<14}{'LORO':>10}" + "".join(f"{'k=%d' % k:>10}" for k in K_PERS))
for a in TEST_ARMS:
    run.log(f"    {a:<14}{np.nanmean(D_LORO[a]):>+10.4f}" +
            "".join(f"{np.nanmean(D_PERS[k][a]):>+10.4f}" for k in K_PERS))

# ── S1 ★★ 주 관문
best_k, best = None, None
for k in K_PERS:
    v = D_PERS[k][GATE_WIN] - D_LORO[GATE_WIN]
    m_, lo_, hi_, n_ = boot_rec_diff(v, SEED0 + 51)
    if n_ >= 3 and (best is None or m_ > best[0]):
        best_k, best = k, (m_, lo_, hi_, n_)
    run.log(f"    (참고) k={k:<3} Δ개인화 − ΔLORO = {m_:+.4f} [{lo_:+.4f}, {hi_:+.4f}]")
if best is None:
    g_("S1", "⛔ 측정 불가", "개인화 대조를 못 냈다")
else:
    m_, lo_, hi_, n_ = best
    eq, sup, nn, md_, frame = judge(m_, lo_, hi_, n_, EQ_DELTA)
    DIFF["S1"] = dict(k=best_k, mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(md_),
                      need_n=(None if nn is None else float(nn)), sup=sup)
    g_("S1", sup, f"★★ **Δ개인화(k={best_k}) − ΔLORO = {m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] "
                  f"· Bonf 3 · {n_}개체")
    run.log(f"       {frame}")

# ── S2 ★★ Δ 의 영점
run.log(f"\n  S2 — Δ 의 **영점**(차원 추가의 0차 비용)")
zero = {a: np.nanmean(D_LORO[a]) for a in CTRL_ARMS}
run.log(f"    대조 팔 LORO Δ — " + " · ".join(f"{a} {zero[a]:+.4f}" for a in CTRL_ARMS))
worst = max(CTRL_ARMS, key=lambda a: zero[a])
v2 = D_LORO[GATE_WIN] - D_LORO[worst]
m2, lo2, hi2, n2 = boot_rec_diff(v2, SEED0 + 52)
if n2 < 3:
    g_("S2", "⛔ 측정 불가", "영점 대조를 못 냈다")
else:
    DIFF["S2"] = dict(ctrl=worst, mean=m2, lo=lo2, hi=hi2, n=n2, mde=float(mde(lo2, hi2)))
    g_("S2", decide(lo2, hi2, 0.0, ">"),
       f"★ `{GATE_WIN}` − `{worst}`(영점) = **{m2:+.4f}** [{lo2:+.4f}, {hi2:+.4f}] "
       f"· MDE {mde(lo2, hi2):.4f} · Bonf 3")
    run.log(f"       ▸ 0 이면 −0.0115 류의 음수는 「정보 없음」이 아니라 **차원 추가 비용**이다")
CONFIG["delta_loro"] = {a: float(np.nanmean(D_LORO[a])) for a in TEST_ARMS}
CONFIG["delta_pers"] = {str(k): {a: float(np.nanmean(D_PERS[k][a])) for a in TEST_ARMS}
                        for k in K_PERS}
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【S-F】 S3 R5 정식화 · S5 필요 표본 · DB 풀링 타당성
run.log("\n" + "=" * 100)
run.log("【S-F】 S3 정식 관문 · S5 필요 표본")
run.log("=" * 100)

# ── S3 ★★ — `f2` 전 계열 잔차화(dr) + 구성 보장 음성 대조 + null 재측정
m3, lo3, hi3, n3 = boot_pair(GATE_WIN, NEG_SHUF, "dr", SEED0 + 61, q=BONF3 * 100)
if n3 < 3:
    g_("S3", "⛔ 측정 불가", "개체 수 부족")
else:
    eq3, sup3, nn3, md3, fr3 = judge(m3, lo3, hi3, n3, EQ_DELTA)
    DIFF["S3"] = dict(mean=m3, lo=lo3, hi=hi3, n=n3, mde=float(md3), sup=sup3,
                      need_n=(None if nn3 is None else float(nn3)))
    g_("S3", sup3, f"★★ [dr · f2 전계열 잔차화] `{GATE_WIN}` − `{NEG_SHUF}` = "
                   f"**{m3:+.4f}** [{lo3:+.4f}, {hi3:+.4f}] · Bonf 3 · {n3}개체")
    run.log(f"       {fr3}")
run.log(f"    (대조) 기존 음성 대조 `{NEG_WIN}` 로 같은 관문 — " + " · ".join(
    f"{m} {boot_pair(GATE_WIN, NEG_WIN, m, SEED0 + 62)[0]:+.4f}" for m in LADDER))
run.log(f"    (프로브 · dr) " + " · ".join(
    f"{a} {np.nanmean(MET['dr'][a])-np.nanmean(NULL['dr'][a]):+.4f}" for a in PROBES))
run.log("    ▸ 프로브가 0 근처면 **`f2` 계열 전체를 덮었다** — 무한회귀 종결의 증거")

# ── S5 ★ 필요 표본 — Q7-R 이 안 센 것
run.log("\n  ★ S5 필요 표본 (R30 ① · Q7-R 은 R1 MDE 0.0145 로 124 레코드면 되는데 안 셌다)")
POOL = {"SVDB": 78, "MIT-BIH": 48, "INCART": 75}
for gname in ("S1", "S2", "S3"):
    d_ = DIFF.get(gname)
    if not d_: continue
    for mar in (EQ_DELTA, 0.02, 0.03):
        nn = need_n(d_["n"], d_["lo"], d_["hi"], d_["mean"], mar)
        txt = "⛔ 불가(점추정이 여유 밖)" if nn is None else f"**{nn:.0f} 레코드**"
        reach = ""
        if nn is not None and np.isfinite(nn):
            cum = 0
            for db, k in POOL.items():
                cum += k
                if cum >= nn:
                    reach = f"  ← {' + '.join(list(POOL)[:list(POOL).index(db)+1])} = {cum} 로 도달"
                    break
            if not reach:
                reach = f"  ← 전체 풀링 {sum(POOL.values())} 로도 부족"
        run.log(f"    {gname} 여유 ±{mar:.2f} → {txt}{reach}")
run.log("    ⚠️ DB 풀링 전제 — 샘플레이트(SVDB 128 · MIT-BIH 360 · INCART 257Hz) 리샘플")
run.log("       규약 고정 · 리드 선택 통일 · **DB 를 고정효과로** 넣고 이질성 보고.")
run.log("       레코드 이질성이 크면 MDE 감소가 √n 보다 **느리다** — 124 가 실제론 180 일 수 있다")
CONFIG["gates"] = VERD; CONFIG["diffs"] = DIFF
run.save_json("config", CONFIG)

In [ ]:
# CELL 9 — 【S-G】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))

# ① ★★ 개인화 vs 교차환자 — 주 가설 그림
xs = [0] + list(range(1, len(K_PERS) + 1))
for a, c_ in zip(TEST_ARMS, ("tab:blue", "tab:red", "tab:gray", "tab:green")):
    ys = [np.nanmean(D_LORO[a])] + [np.nanmean(D_PERS[k][a]) for k in K_PERS]
    ax[0].plot(xs, ys, "o-", color=c_, label=a)
ax[0].axhline(0, color="k", lw=.8)
ax[0].set_xticks(xs); ax[0].set_xticklabels(["LORO"] + [f"k={k}" for k in K_PERS], fontsize=8)
ax[0].set_ylabel("delta AUPRC vs rhythm-only")
ax[0].set_xlabel("cross-patient  ->  personalized")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② SNR 선택
ms = list(LADDER)
ax[1].bar(range(len(ms)), [SNR[m] for m in ms],
          color=["tab:green" if m == BEST else "tab:gray" for m in ms])
ax[1].set_xticks(range(len(ms))); ax[1].set_xticklabels(ms, rotation=30, fontsize=8)
ax[1].set_ylabel(f"SNR = recovery / MDE  (mean over a in {list(TARGET_A)})")
ax[1].set_xlabel("estimator  (higher is better)"); ax[1].grid(alpha=.3, axis="y")

# ③ 관문 CI
gs = [g for g in ("S1", "S2", "S3") if g in DIFF]
ys = np.arange(len(gs))
if gs:
    mm_ = [DIFF[g]["mean"] for g in gs]
    lo_ = [DIFF[g]["lo"] for g in gs]; hi_ = [DIFF[g]["hi"] for g in gs]
    ax[2].errorbar(mm_, ys, xerr=[np.array(mm_) - np.array(lo_),
                                  np.array(hi_) - np.array(mm_)],
                   fmt="o", color="tab:blue", capsize=4)
    ax[2].set_yticks(ys); ax[2].set_yticklabels(gs, fontsize=9)
ax[2].axvline(0, color="k", lw=.8)
ax[2].axvspan(-EQ_DELTA, EQ_DELTA, color="tab:green", alpha=.15)
ax[2].set_xlabel(f"gate estimate  (shaded = equivalence +-{EQ_DELTA})")
ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7s_personalize", fig)
plt.close(fig)
display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")   # R29 ②
run.log(f"  검출 규칙 — {DETECT_RULE}")
run.log(f"  주 추정량(SNR 선택) — **`{BEST}`** · 격자 감쌈 {'✅' if BRACKET else '⛔ 미확정'}")
run.log(f"  누출 바닥(부트스트랩) {LEAK_PT:+.4f} [{LEAK_LO:+.4f}, {LEAK_HI:+.4f}]")
for g in ("S1", "S2", "S3"):
    d_ = DIFF.get(g)
    if d_ is None:
        run.log(f"  {g:<4}⛔ 측정 불가"); continue
    lim = "⛔ MDE 아래" if abs(d_["mean"]) < d_["mde"] else "▸ MDE 위"
    run.log(f"  {g:<4}{VERD.get(g)} · {d_['mean']:+.4f} [{d_['lo']:+.4f}, {d_['hi']:+.4f}] "
            f"· MDE {d_['mde']:.4f} · {lim}")
run.log("")
if any(un_(g) or g not in DIFF for g in ("S1", "S2", "S3")):
    run.log("  ⛔ 측정 불가가 있다 — 어떤 결론 분기도 타지 않는다(R29 ②)")
elif ok_("S1") and ok_("S2"):
    run.log("  ★★ **H 확증 — 창 형태는 개인 내에만 있고 교차환자로 전이되지 않는다.**")
    run.log("     Q7 은 부정 결과가 아니라 **개인화 아키텍처의 근거**다. 이 문장으로 종결하고")
    run.log("     V 시리즈(소량 라벨 개인화)·R7(라벨프리 적응 임계)로 넘긴다")
elif ok_("S1") and not ok_("S2"):
    run.log("  ⚠️ **개인화 이득이 창 내용이 아니라 차원 추가에서 온다** — 영점 대조가 이겼다.")
    run.log("     창 특징 표현(PCA)을 바꾸거나 차원을 맞춘 대조로 다시 묻는다")
elif not ok_("S1") and ok_("S3"):
    run.log("  ⚠️ **개체 내 신호는 있는데 소량 라벨로는 못 꺼낸다.** `k` 를 늘리거나")
    run.log("     표현을 바꾼다(템플릿 대신 정합 필터 등)")
else:
    nn1 = DIFF.get("S1", {}).get("need_n")
    if nn1 is None or (np.isfinite(nn1) and nn1 > 300):
        run.log("  ⛔ **사전등록 종결 조건 발동**(R34 ⑤) — S1·S3 미결이고 필요 표본이 300 초과다.")
        run.log("     「**SVDB 규모에서 P 파 고유 신호는 측정 불가능하다**」를 퀘스트 최종")
        run.log("     문장으로 쓰고 **형태 갈래를 닫는다. 더 돌지 않는다.**")
    else:
        run.log(f"  ⚠️ 미결이지만 필요 표본 {nn1:.0f} 레코드로 **도달 가능**하다 — DB 풀링으로 간다")

run.finish({
    "exp_id": "quest46_q7s_personalize",
    "metric": "svdb_delta_personalized_minus_loro",
    "value": float(DIFF.get("S1", {}).get("mean", float("nan"))),
    "passed": bool(ok_("S1") and ok_("S2")),
    "summary": ("개인 내 vs 교차환자를 주 가설로 승격하고, Q7-R 의 교정 결함 넷"
                "(검출 규칙·선택 기준·격자·누출 바닥 선택편의)을 코드로 고쳤다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "detect_rule": DETECT_RULE, "estimator": CONFIG.get("estimator", {}),
    "best_estimator": BEST, "bracketed": bool(BRACKET), "leak": CONFIG.get("leak", {}),
    "delta_loro": CONFIG.get("delta_loro", {}), "delta_pers": CONFIG.get("delta_pers", {}),
    "excess": CONFIG.get("excess", {}), "n_scored": len(RS), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-personalize`")